# Visualize Best Agent

This notebook loads the best agent from a pickle file, displays its program code, and visualizes it playing FlappyBird with human rendering.


In [1]:
# Import required libraries
import flappy_bird_env  # noqa
import numpy as np
import os
import pickle
from pathlib import Path
from scipy.special import expit

# Disable headless mode for human rendering
os.environ.pop('SDL_VIDEODRIVER', None)

try:
    import pygame
    pygame.init()
    print("✓ Pygame initialized for human rendering")
except Exception as e:
    print(f"Warning: Could not initialize pygame: {e}")

import gymnasium as gym
from memory_system import MemoryConfig, MemoryType
from evaluator import FlappyBirdEvaluator, FlappyBirdEvaluatorConfig


✓ Pygame initialized for human rendering


## Load Best Agent

Load the best agent pickle file. This file contains just the Individual object.


In [2]:
# Path to best agent file (adjust if needed)
agent_path = "best_agent_gen0453_fit1_0980.pkl"

# Check if file exists
if not Path(agent_path).exists():
    print(f"❌ Agent file not found: {agent_path}")
    print("Please update agent_path to point to your best agent pickle file.")
    best_agent = None
else:
    print(f"Loading best agent from: {agent_path}")
    
    # Load the Individual directly (best agent files contain just the Individual object)
    with open(agent_path, 'rb') as f:
        best_agent = pickle.load(f)
    
    # Extract generation from filename if possible
    # Filename format: best_agent_gen{XXXX}_fit{X_XXXX}.pkl
    filename = Path(agent_path).stem
    try:
        parts = filename.split('_')
        gen_part = [p for p in parts if p.startswith('gen')][0]
        generation = int(gen_part.replace('gen', ''))
    except:
        generation = "unknown"
    
    print(f"✓ Best agent loaded successfully!")
    print(f"  Agent ID: {best_agent.id}")
    print(f"  Generation (from filename): {generation}")
    if best_agent.fitness is not None:
        print(f"  Fitness: {best_agent.fitness:.4f}")
    else:
        print(f"  Fitness: not set")
    print(f"  Program length: {len(best_agent.program)}")
    print(f"  Age: {best_agent.age}")
    if best_agent.parent_ids:
        print(f"  Parent IDs: {best_agent.parent_ids}")


Loading best agent from: best_agent_gen0453_fit1_0980.pkl
✓ Best agent loaded successfully!
  Agent ID: 136320
  Generation (from filename): 453
  Fitness: 1.0980
  Program length: 15
  Age: 0
  Parent IDs: (55660, 84863)


## Display Program Code

Show the full program and the effective program (with introns removed).


In [3]:
# Reconstruct MemoryConfig from the agent's MemoryBank
# This is needed for instruction string formatting
memory = best_agent.memory
memory_cfg = MemoryConfig(
    n_scalar=8,
    n_vector=8,
    n_matrix=8,
    n_obs_scalar=0,
    n_obs_vector=1,
    n_obs_matrix=1,
    vector_size=64,
    matrix_shape=(64,64),
)

# Output registers (default for FlappyBird, adjust if your config uses different)
output_registers = [(MemoryType.SCALAR, 0)]

print("="*80)
print("FULL PROGRAM")
print("="*80)
print()

for i, instr in enumerate(best_agent.program.instructions):
    print(f"{i:4d}: {instr.to_resolved_str(memory_cfg)}")

print()
print("="*80)
print(f"Total: {len(best_agent.program)} instructions")
print("="*80)


FULL PROGRAM

   0: scalar[2→2] = scalar_div_protected(obs_vector[0→0][8458→10], scalar[8929→1])
   1: scalar[4→4] = automl_scalar_log(scalar[3482→2])
   2: scalar[0→0] = automl_scalar_exp(obs_matrix[0→0][44,6])
   3: scalar[6→6] = scalar_mul(scalar[417→1], scalar[4814→6])
   4: scalar[7→7] = scalar_conditional(obs_matrix[0→0][14,22], obs_matrix[0→0][29,63])
   5: scalar[6→6] = scalar_mul(obs_matrix[0→0][59,59], obs_vector[0→0][9245→29])
   6: scalar[7→7] = scalar_mul(scalar[5164→4], scalar[2681→1])
   7: scalar[5→5] = automl_scalar_exp(scalar[2405→5])
   8: scalar[2→2] = scalar_mul(obs_matrix[0→0][46,30], obs_vector[0→0][746→42])
   9: scalar[6→6] = scalar_mul(scalar[5987→3], scalar[3264→0])
  10: scalar[0→0] = automl_scalar_cos(scalar[323→3])
  11: scalar[1→1] = scalar_sub(scalar[2411→3], obs_vector[0→0][220→28])
  12: scalar[3→3] = scalar_mul(obs_matrix[0→0][10,61], scalar[5733→5])
  13: scalar[5→5] = scalar_conditional(scalar[580→4], obs_matrix[0→0][10,46])
  14: scalar[6→6] = scal

In [4]:
# Show effective program (with introns removed)
effective_program = best_agent.get_effective_program(output_registers)

print("="*80)
print("EFFECTIVE PROGRAM (INTRONS REMOVED)")
print("="*80)
print()

if len(effective_program.instructions) > 0:
    for i, instr in enumerate(effective_program.instructions):
        print(f"{i:4d}: {instr.to_resolved_str(memory_cfg)}")
else:
    print("  (No effective instructions found)")

print()
print("="*80)
print(f"Effective: {len(effective_program)} instructions (out of {len(best_agent.program)} total)")
if len(best_agent.program) > 0:
    effective_ratio = len(effective_program) / len(best_agent.program)
    print(f"Effective code rate: {effective_ratio:.3f} ({effective_ratio*100:.1f}%)")
print("="*80)


EFFECTIVE PROGRAM (INTRONS REMOVED)

   0: scalar[0→0] = automl_scalar_cos(scalar[323→3])

Effective: 1 instructions (out of 15 total)
Effective code rate: 0.067 (6.7%)


## Create Evaluator with Human Rendering

Set up the FlappyBird evaluator with human rendering mode to visualize the agent playing.


In [5]:
# Get the expected vector size from the agent's memory (for feature_vector strategy)
feature_vector_size = best_agent.memory.vector_size
print(f"Agent's vector size: {feature_vector_size}")
print(f"Agent's matrix shape: {best_agent.memory.matrix_shape}")

# Create evaluator config for visualization
# Use the same config as training, but with human rendering
evaluator_config = FlappyBirdEvaluatorConfig(
    env_id="FlappyBird-v0",
    episodes=3,  # Run 3 episodes to see the agent play
    max_steps=500,
    output_register=0,
    render_mode="human",  # Human rendering to see the game
    rng_seed=42,
    patch_strategy="feature_vector",  # Using feature_vector strategy
    color_channel=2,  # Not used for feature_vector, but required
    normalize=True,
    quantization_factor=0.5,  # Not used for feature_vector
    feature_vector_size=feature_vector_size,  # Match agent's vector size
    output_registers=[(MemoryType.SCALAR, 0)],
    n_jobs=1,  # Sequential for visualization
)

print("\nCreating FlappyBird evaluator with human rendering...")
evaluator = FlappyBirdEvaluator(config=evaluator_config)
print("✓ Evaluator created!")
print(f"  Episodes: {evaluator.episodes}")
print(f"  Max steps per episode: {evaluator.max_steps}")
print(f"  Render mode: {evaluator.config.render_mode}")
print(f"  Patch strategy: feature_vector")
print(f"  Feature vector size: {feature_vector_size}")
print()
print("⚠️  NOTE: FlappyBird windows will appear when you run the next cell!")
print("   Close the windows or press Ctrl+C to stop.")


Agent's vector size: 64
Agent's matrix shape: (64, 64)

Creating FlappyBird evaluator with human rendering...
✓ Evaluator created!
  Episodes: 3
  Max steps per episode: 500
  Render mode: human
  Patch strategy: feature_vector
  Feature vector size: 64

⚠️  NOTE: FlappyBird windows will appear when you run the next cell!
   Close the windows or press Ctrl+C to stop.


## Visualize Agent Playing

Run the agent and watch it play FlappyBird. The game windows will appear showing the agent's performance.


In [ ]:
# Run the agent and visualize
print("="*80)
print("RUNNING BEST AGENT")
print("="*80)
print()

total_reward = 0.0
episode_rewards = []

for episode_idx in range(evaluator.episodes):
    print(f"\nEpisode {episode_idx + 1}/{evaluator.episodes}")
    print("-" * 80)
    
    # Seed the episode
    episode_seed = int((evaluator.config.rng_seed + episode_idx * 100) % (2**31))
    observation, _ = evaluator.env.reset(seed=episode_seed)
    observation = np.asarray(observation, dtype=np.float32)
    
    # Copy memory for this episode
    memory = best_agent.memory.copy()
    episode_reward = 0.0
    steps = 0
    
    for step in range(evaluator.max_steps):
        # Process observation
        # Note: _process_observation returns different formats based on strategy:
        # - feature_vector: returns dict {'vector': [...], 'matrix': [...]}
        # - quantized/full_image: returns tuple (observations_list, 'matrix')
        result = evaluator._process_observation(observation)
        
        # Load observations into memory
        if isinstance(result, dict):
            # feature_vector strategy returns both vector and matrix
            memory.load_observation(result)
        else:
            # quantized/full_image strategy returns tuple
            processed_observations, obs_type = result
            if obs_type == 'vector':
                memory.load_observation({'vector': processed_observations})
            else:
                memory.load_observation({'matrix': processed_observations})
        
        # Execute the program
        best_agent.program.execute(memory,debug=True)
        
        # Read action from output register
        action_value = memory.read_scalar(evaluator.output_register)
        normalized = expit(action_value)  # Sigmoid
        action = 1 if normalized >= 0.5 else 0
        
        # Take step in environment
        observation, reward, terminated, truncated, _ = evaluator.env.step(action)
        observation = np.asarray(observation, dtype=np.float32)
        episode_reward += reward
        steps += 1
        
        if terminated or truncated:
            break
    
    episode_rewards.append(episode_reward)
    total_reward += episode_reward
    
    print(f"  Steps: {steps}")
    print(f"  Reward: {episode_reward:.2f}")
    print(f"  Action value (scalar[0]): {action_value:.4f}")
    print(f"  Normalized (sigmoid): {normalized:.4f}")
    print(f"  Action chosen: {'FLAP' if action == 1 else 'NOOP'}")

print()
print("="*80)
print("SUMMARY")
print("="*80)
print(f"Total episodes: {evaluator.episodes}")
print(f"Average reward: {total_reward / evaluator.episodes:.2f}")
print(f"Rewards per episode: {[f'{r:.2f}' for r in episode_rewards]}")
print("="*80)

# Close the evaluator
evaluator.close()
print("\n✓ Visualization complete!")


RUNNING BEST AGENT


Episode 1/3
--------------------------------------------------------------------------------


KeyboardInterrupt: 

: 

## Additional Information

Display additional details about the best agent's memory and constants.


In [7]:
# Display memory information
print("="*80)
print("BEST AGENT MEMORY INFORMATION")
print("="*80)
print()

memory = best_agent.memory

print("Scalar registers (working):")
for i in range(min(8, memory.n_scalar)):
    print(f"  scalar[{i}]: {memory.scalars[i]:.6f}")

print()
print("Vector registers (working):")
for i in range(min(3, memory.n_vector)):
    vec_str = ", ".join([f"{v:.3f}" for v in memory.vectors[i][:5]])
    if len(memory.vectors[i]) > 5:
        vec_str += "..."
    print(f"  vector[{i}]: [{vec_str}]")

print()
print("Matrix registers (working):")
for i in range(min(3, memory.n_matrix)):
    print(f"  matrix[{i}]: shape {memory.matrices[i].shape}, "
          f"mean={memory.matrices[i].mean():.4f}, "
          f"std={memory.matrices[i].std():.4f}")

print()
print("Observation registers:")
if memory.n_obs_matrix > 0:
    print(f"  obs_matrix[0]: shape {memory.obs_matrices[0].shape}")

print("="*80)


BEST AGENT MEMORY INFORMATION

Scalar registers (working):
  scalar[0]: -8.803028
  scalar[1]: -3.145897
  scalar[2]: 2.946764
  scalar[3]: -0.510127
  scalar[4]: 2.707772
  scalar[5]: 1.814998
  scalar[6]: -1.715441
  scalar[7]: 0.376902

Vector registers (working):
  vector[0]: [0.117, 0.664, 0.813, -0.341, -0.756...]
  vector[1]: [0.176, 0.096, -2.005, -0.451, -1.252...]
  vector[2]: [1.340, 1.065, -0.591, 1.108, -2.091...]

Matrix registers (working):
  matrix[0]: shape (64, 64), mean=-0.0016, std=0.8907
  matrix[1]: shape (64, 64), mean=0.0470, std=0.8772
  matrix[2]: shape (64, 64), mean=-0.0130, std=0.9086

Observation registers:
  obs_matrix[0]: shape (64, 64)
